# 02 · Data Preparation (Bagian Hafid)
**Proyek:** Prediksi Harga Harian Komoditas Pangan (SDG 2, Target 2.c) · **Minggu 2** [L] Data Understanding & Data Preparation

**Bagian notebook ini (PIC: Hafid):**

| Bagian | Isi | Bab laporan |
|---|---|---|
| A | EDA minimal 3 visualisasi + insight (hanya pada data latih) | 4.4 |
| B | Feature engineering: lag, rolling, kalender, hari raya | 5.2 |
| C | Transformasi: scaling dan jendela waktu untuk LSTM | 5.3 |
| D | Uji anti-leakage dan pencatatan jumlah sampel tiap split | checklist |
| E | Ringkasan Apa-Mengapa-Bagaimana | 5.6 |

**Prasyarat dari Rehan** (cleaning dan split, tugas 4a dan 6): berkas `data/processed/train.csv` dan `data/processed/val.csv` dengan kolom tanggal dan harga, terurut kronologis, sudah bersih. Sepakati nama berkas dan kolom, lalu sesuaikan di sel Konfigurasi.

**Aturan anti-leakage yang dipegang notebook ini:**
1. **Test set tidak dimuat sama sekali.** Notebook hanya membaca train dan validation.
2. EDA dan keputusan fitur hanya memakai data latih.
3. Setiap fitur pada tanggal *t* hanya memakai harga sampai tanggal *t*.
4. Scaler di-fit **hanya** pada data latih.
5. Baris dimasukkan ke split berdasarkan **tanggal targetnya**, bukan tanggal fiturnya.

## 0. Setup dan Konfigurasi

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.preprocessing import MinMaxScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (11, 4)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "models"
FIG_DIR = ROOT / "reports" / "figures"
for d in (DATA_DIR, MODEL_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
# ===== KONFIGURASI (sesuaikan dengan keluaran Rehan) =====
USE_DEMO_DATA = False      # True hanya untuk mencoba notebook tanpa data asli (DATA SINTETIS!)
DATE_COL = "tanggal"       # nama kolom tanggal
PRICE_COL = "harga"        # nama kolom harga (Rp/kg)
TRAIN_FILE = "train.csv"
VAL_FILE = "val.csv"

LAGS = [1, 2, 7]           # dalam satuan OBSERVASI. Jika data hanya hari kerja, 1 minggu = 5 observasi
ROLL_WINDOWS = [7, 14]     # jendela rata-rata dan simpangan baku bergulir
HORIZON = 1                # memprediksi harga hari berikutnya
WINDOW = 14                # panjang jendela LSTM (rencana: 14 sampai 30)
LSTM_EXTRA_COLS = []       # opsional (multivariat), mis. ["month_sin", "month_cos", "dekat_hari_raya"]
HARI_RAYA_TOL = 7          # fitur 'dekat_hari_raya' = 1 jika selisih <= 7 hari

# Kalender hari raya (VERIFIKASI dan lengkapi sesuai rentang data Anda, termasuk 2026 dan Idul Adha)
IDUL_FITRI = ["2022-05-02", "2023-04-22", "2024-04-10", "2025-03-31"]
_years = range(2018, 2030)
_fixed = [f"{y}-01-01" for y in _years] + [f"{y}-12-25" for y in _years]
HARI_RAYA = pd.DatetimeIndex(sorted(pd.to_datetime(IDUL_FITRI + _fixed)))

def dekat_hari_raya(dates, tol=HARI_RAYA_TOL):
    """1 jika tanggal berjarak <= tol hari dari hari raya terdekat. Kalender diketahui sebelumnya, jadi bukan leakage."""
    d = pd.DatetimeIndex(dates)
    diff = np.abs((d.values[:, None] - HARI_RAYA.values[None, :]).astype("timedelta64[D]").astype(int))
    return (diff.min(axis=1) <= tol).astype(int)

## 1. Memuat Data Latih dan Validasi
Test set sengaja **tidak dimuat** sampai evaluasi akhir.

In [ ]:
def make_demo():
    """Data SINTETIS hanya untuk menguji alur notebook. Jangan dipakai untuk laporan."""
    rng = np.random.default_rng(SEED)
    dates = pd.date_range("2022-01-01", "2025-12-31", freq="D")
    t = np.arange(len(dates))
    price = 45000 + 5 * t + 6000 * np.sin(2 * np.pi * t / 365.25) + np.cumsum(rng.normal(0, 400, len(dates)))
    near = dekat_hari_raya(dates, tol=10)
    price = price + 9000 * near
    df = pd.DataFrame({DATE_COL: dates, PRICE_COL: np.round(price, -2)})
    n_train, n_val = int(0.70 * len(df)), int(0.15 * len(df))
    return df.iloc[:n_train].copy(), df.iloc[n_train:n_train + n_val].copy()   # bagian test dibuang

def read_split(name):
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"{path} belum ada. Minta Rehan menyimpan hasil split ke data/processed/, "
                                f"atau set USE_DEMO_DATA = True hanya untuk mencoba alur.")
    df = pd.read_csv(path, parse_dates=[DATE_COL])
    return df[[DATE_COL, PRICE_COL]].sort_values(DATE_COL).reset_index(drop=True)

if USE_DEMO_DATA:
    print("PERINGATAN: memakai DATA SINTETIS. Jangan dipakai untuk laporan.")
    train_raw, val_raw = make_demo()
else:
    train_raw, val_raw = read_split(TRAIN_FILE), read_split(VAL_FILE)
train_raw = train_raw.reset_index(drop=True)
val_raw = val_raw.reset_index(drop=True)

full = pd.concat([train_raw, val_raw], ignore_index=True)
TRAIN_END, VAL_END = train_raw[DATE_COL].max(), val_raw[DATE_COL].max()

assert full[DATE_COL].is_monotonic_increasing, "Tanggal harus terurut"
assert not full[DATE_COL].duplicated().any(), "Ada tanggal ganda"
assert full[PRICE_COL].notna().all(), "Ada harga kosong, minta Rehan melengkapi cleaning"
assert train_raw[DATE_COL].max() < val_raw[DATE_COL].min(), "Train dan val tumpang tindih"

print(f"Train: {len(train_raw)} baris, {train_raw[DATE_COL].min().date()} s/d {TRAIN_END.date()}")
print(f"Val  : {len(val_raw)} baris, {val_raw[DATE_COL].min().date()} s/d {VAL_END.date()}")
print("Selisih antar tanggal (hari):")
print(full[DATE_COL].diff().dt.days.dropna().value_counts().head())

## A. EDA pada Data Latih (Bab 4.4)
Seluruh EDA memakai **data latih saja** agar keputusan fitur tidak dipengaruhi data validasi. Setelah melihat tiap grafik, isi sel *Insight* dengan temuan Anda sendiri.

In [ ]:
# Visualisasi 1: tren harga
fig, ax = plt.subplots()
ax.plot(train_raw[DATE_COL], train_raw[PRICE_COL], lw=0.8, label="harga harian")
ax.plot(train_raw[DATE_COL], train_raw[PRICE_COL].rolling(30, min_periods=1).mean(), lw=2, label="rata-rata 30 observasi")
ax.set_title("Tren harga (data latih)"); ax.set_ylabel("Rp/kg"); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "eda_01_tren.png", dpi=150); plt.show()
display(train_raw[PRICE_COL].describe().to_frame().T)

**Insight 1 (isi):** Bagaimana tren jangka panjangnya? Apakah ada lonjakan besar? Apakah variansinya berubah seiring waktu?

In [ ]:
# Visualisasi 2: distribusi harga dan pola musiman per bulan
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train_raw[PRICE_COL], bins=40)
axes[0].set_title("Distribusi harga"); axes[0].set_xlabel("Rp/kg")
tmp = train_raw.assign(bulan=train_raw[DATE_COL].dt.month)
tmp.boxplot(column=PRICE_COL, by="bulan", ax=axes[1], grid=False)
axes[1].set_title("Harga per bulan"); axes[1].set_xlabel("bulan"); fig.suptitle("")
plt.tight_layout(); plt.savefig(FIG_DIR / "eda_02_distribusi_musiman.png", dpi=150); plt.show()

**Insight 2 (isi):** Apakah distribusi miring ke kanan (ekor panjang)? Bulan mana yang cenderung tinggi? Apa implikasinya bagi fitur bulan dan pemilihan metrik?

In [ ]:
# Visualisasi 3: autokorelasi (ACF) untuk memilih lag
max_lag = 60
acf = [train_raw[PRICE_COL].autocorr(lag=k) for k in range(1, max_lag + 1)]
band = 1.96 / np.sqrt(len(train_raw))
fig, ax = plt.subplots()
ax.bar(range(1, max_lag + 1), acf)
ax.axhline(band, ls="--", c="gray"); ax.axhline(-band, ls="--", c="gray")
ax.set_title("Autokorelasi harga (data latih)"); ax.set_xlabel("lag (observasi)"); ax.set_ylabel("korelasi")
plt.tight_layout(); plt.savefig(FIG_DIR / "eda_03_acf.png", dpi=150); plt.show()
print({k: round(v, 3) for k, v in zip([1, 2, 7, 14, 30], [acf[0], acf[1], acf[6], acf[13], acf[29]])})

**Insight 3 (isi):** Seberapa kuat korelasi pada lag 1, 7, dan 14? Apakah pilihan `LAGS` dan `ROLL_WINDOWS` perlu diubah?

In [ ]:
# Visualisasi 4 (tambahan): lonjakan harga dan kedekatannya dengan hari raya
ret = train_raw[PRICE_COL].pct_change()
spk = train_raw.assign(perubahan=ret, dekat_hari_raya=dekat_hari_raya(train_raw[DATE_COL]))
spk = spk[spk["perubahan"].abs() > 3 * ret.std()]

fig, ax = plt.subplots()
ax.plot(train_raw[DATE_COL], train_raw[PRICE_COL], lw=0.8)
ax.scatter(spk[DATE_COL], spk[PRICE_COL], c="red", s=18, label="lonjakan > 3 simpangan baku")
for h in HARI_RAYA[(HARI_RAYA >= train_raw[DATE_COL].min()) & (HARI_RAYA <= TRAIN_END)]:
    ax.axvline(h, c="orange", alpha=0.3)
ax.set_title("Lonjakan harga dan hari raya (garis oranye)"); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR / "eda_04_lonjakan.png", dpi=150); plt.show()
print(f"Jumlah lonjakan: {len(spk)}, di antaranya dekat hari raya: {int(spk['dekat_hari_raya'].sum())}")
display(spk.head(10))

**Insight 4 (isi):** Berapa persen lonjakan yang dekat hari raya? Apakah lonjakan layak dipertahankan (kemungkinan nyata) dan bukan dibuang sebagai outlier?

## B. Feature Engineering (Bab 5.2)
**Konvensi:** baris pada tanggal *t* memprediksi harga pada *t + HORIZON*. Semua fitur hanya memakai harga sampai *t*:
- `harga_t`, `lag_k`: harga *k* observasi sebelumnya.
- `roll_mean_w`, `roll_std_w`: statistik jendela yang **berakhir di t** (tidak melihat masa depan).
- Kalender: hari dalam minggu, bulan (juga bentuk siklik sin/cos), dan `dekat_hari_raya`.

Fitur dibangun pada rangkaian train+val **tanpa menyentuh test**. Nilai fitur val boleh memakai data latih sebagai masa lalu, sama seperti saat model dipakai nyata.

In [ ]:
def build_supervised(series_df):
    df = series_df[[DATE_COL, PRICE_COL]].copy().reset_index(drop=True)
    p, d = df[PRICE_COL], df[DATE_COL]
    out = pd.DataFrame({DATE_COL: d, "harga_t": p})
    for k in LAGS:
        out[f"lag_{k}"] = p.shift(k)
    for w in ROLL_WINDOWS:
        out[f"roll_mean_{w}"] = p.rolling(w).mean()
        out[f"roll_std_{w}"] = p.rolling(w).std()
    out["selisih_1"] = p.diff(1)
    out["dow"] = d.dt.dayofweek
    out["month"] = d.dt.month
    out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12)
    out["dekat_hari_raya"] = dekat_hari_raya(d)
    out["target_date"] = d.shift(-HORIZON)
    out["y"] = p.shift(-HORIZON)
    return out

raw_sup = build_supervised(full)
sup = raw_sup.dropna().reset_index(drop=True)
print(f"Baris dibuang karena warm-up lag/rolling dan akhir deret: {len(raw_sup) - len(sup)}")

# Split berdasarkan TANGGAL TARGET: target train tidak pernah melewati TRAIN_END
sup["split"] = np.where(sup["target_date"] <= TRAIN_END, "train",
                np.where(sup["target_date"] <= VAL_END, "val", "test"))
assert (sup["split"] != "test").all(), "Ada baris berstatus test, periksa input"

NON_FEATURES = [DATE_COL, "target_date", "y", "split"]
FEATURE_COLS = [c for c in sup.columns if c not in NON_FEATURES]
print("Fitur:", FEATURE_COLS)
display(sup.head())

## C. Transformasi (Bab 5.3)
- **Scaling (MinMax):** untuk Linear Regression dan LSTM. XGBoost tidak membutuhkannya, jadi tabel tanpa scaling juga disimpan.
- **Scaler di-fit hanya pada baris train.** Nilai val boleh keluar dari rentang [0, 1] (harga naik di luar rentang latih). Itu wajar dan **tidak** di-clip.
- **Jendela LSTM:** tiap sampel berisi `WINDOW` baris berurutan dengan bentuk `(sampel, WINDOW, fitur)`.

In [ ]:
train_mask = sup["split"] == "train"

x_scaler = MinMaxScaler().fit(sup.loc[train_mask, FEATURE_COLS])
y_scaler = MinMaxScaler().fit(sup.loc[train_mask, ["y"]])

sup_scaled = sup.copy()
sup_scaled[FEATURE_COLS] = x_scaler.transform(sup[FEATURE_COLS])
sup_scaled["y"] = y_scaler.transform(sup[["y"]]).ravel()

joblib.dump(x_scaler, MODEL_DIR / "x_scaler.joblib")
joblib.dump(y_scaler, MODEL_DIR / "y_scaler.joblib")

# Tabel untuk model klasik (tanpa dan dengan scaling)
for name in ("train", "val"):
    sup[sup["split"] == name].to_csv(DATA_DIR / f"features_{name}.csv", index=False)
    sup_scaled[sup_scaled["split"] == name].to_csv(DATA_DIR / f"features_scaled_{name}.csv", index=False)
print("Tabel fitur tersimpan di", DATA_DIR)

In [ ]:
LSTM_COLS = ["harga_t"] + [c for c in LSTM_EXTRA_COLS if c != "harga_t"]

def make_windows(frame, cols, window):
    A = frame[cols].to_numpy(dtype="float32")
    y = frame["y"].to_numpy(dtype="float32")
    X = np.stack([A[i - window + 1: i + 1] for i in range(window - 1, len(A))])
    meta = frame.iloc[window - 1:][[DATE_COL, "target_date", "split"]].reset_index(drop=True)
    return X, y[window - 1:], meta

X_all, y_all, meta = make_windows(sup_scaled, LSTM_COLS, WINDOW)
m_tr, m_va = (meta["split"] == "train").to_numpy(), (meta["split"] == "val").to_numpy()
X_train, y_train, X_val, y_val = X_all[m_tr], y_all[m_tr], X_all[m_va], y_all[m_va]

np.savez_compressed(DATA_DIR / f"lstm_windows_W{WINDOW}.npz",
                    X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val,
                    target_date_train=meta.loc[m_tr, "target_date"].astype(str).to_numpy(),
                    target_date_val=meta.loc[m_va, "target_date"].astype(str).to_numpy())
print("Bentuk X_train:", X_train.shape, "| X_val:", X_val.shape)

## D. Uji Anti-Leakage dan Jumlah Sampel (checklist)
Sel ini berhenti dengan error jika ada pelanggaran. Jalankan ulang setiap kali pipeline diubah.

In [ ]:
tr, va = sup[sup["split"] == "train"], sup[sup["split"] == "val"]

# 1. split berdasarkan tanggal target dan tidak tumpang tindih
assert tr["target_date"].max() <= TRAIN_END
assert va["target_date"].min() > TRAIN_END and va["target_date"].max() <= VAL_END
assert set(tr["target_date"]).isdisjoint(set(va["target_date"]))

# 2. target selalu di masa depan relatif terhadap fitur
assert (sup["target_date"] > sup[DATE_COL]).all()
assert "y" not in FEATURE_COLS and "target_date" not in FEATURE_COLS

# 3. scaler hanya melihat data train
assert np.allclose(x_scaler.data_min_, tr[FEATURE_COLS].min().to_numpy())
assert np.allclose(x_scaler.data_max_, tr[FEATURE_COLS].max().to_numpy())
assert np.allclose(y_scaler.data_min_, tr["y"].min()) and np.allclose(y_scaler.data_max_, tr["y"].max())

# 4. spot-check lag dan target terhadap data asli
pos = {dt: i for i, dt in enumerate(full[DATE_COL])}
for _, row in sup.sample(min(30, len(sup)), random_state=SEED).iterrows():
    i = pos[row[DATE_COL]]
    for k in LAGS:
        assert row[f"lag_{k}"] == full.loc[i - k, PRICE_COL]
    assert row["y"] == full.loc[i + HORIZON, PRICE_COL]
    assert row["harga_t"] == full.loc[i, PRICE_COL]

# 5. jendela LSTM: setiap sampel hanya berisi masa lalu
assert X_train.shape[1] == WINDOW and not np.isnan(X_train).any() and not np.isnan(X_val).any()

print("Semua uji anti-leakage lulus.")

In [ ]:
# Jumlah sampel tiap split (dicatat untuk Bab 5.4 dan checklist)
counts = pd.DataFrame({
    "split": ["train", "val", "test (tidak dimuat)"],
    "baris_mentah": [len(train_raw), len(val_raw), "-"],
    "sampel_tabular": [len(tr), len(va), "-"],
    "sampel_lstm": [len(X_train), len(X_val), "-"],
    "tanggal_target_awal": [tr["target_date"].min().date(), va["target_date"].min().date(), "-"],
    "tanggal_target_akhir": [tr["target_date"].max().date(), va["target_date"].max().date(), "-"],
})
counts.to_csv(ROOT / "reports" / "sample_counts_hafid.csv", index=False)
display(counts)

## E. Ringkasan Apa–Mengapa–Bagaimana (Bab 5.6)
Tabel di bawah untuk langkah Hafid. Rehan menambahkan baris untuk cleaning (5.1), split (5.4), dan balancing (5.5), lalu Hafid menggabungkannya menjadi satu tabel akhir.

| Langkah | Apa | Mengapa | Bagaimana |
|---|---|---|---|
| EDA hanya pada data latih | Visualisasi tren, distribusi, ACF, dan lonjakan | Keputusan fitur tidak boleh dipengaruhi data validasi | Semua grafik memakai `train_raw` |
| Fitur lag | `harga_t`, `lag_1`, `lag_2`, `lag_7` | Harga berkorelasi dengan harga hari-hari sebelumnya (lihat ACF) | `shift(k)` pada deret harga |
| Statistik bergulir | Rata-rata dan simpangan baku jendela 7 dan 14 | Menangkap tren dan volatilitas jangka pendek | `rolling(w)` yang berakhir di *t*, tanpa data masa depan |
| Selisih harian | `selisih_1` | Menunjukkan arah dan besar perubahan terakhir | `diff(1)` |
| Fitur kalender | Hari dalam minggu, bulan, sin/cos bulan | Harga memiliki pola musiman | Diambil dari tanggal, sin/cos agar Desember dan Januari berdekatan |
| Hari raya | `dekat_hari_raya` | Lonjakan harga sering mendekati hari raya | Selisih ke hari raya terdekat <= 7 hari. Kalender diketahui sebelumnya |
| Penentuan split baris | Setiap baris masuk split menurut tanggal target | Target latih tidak boleh berasal dari periode validasi | Perbandingan `target_date` dengan `TRAIN_END` |
| Scaling MinMax | Fitur dan target diskalakan ke [0, 1] | Linear Regression dan LSTM sensitif terhadap skala | `MinMaxScaler.fit` hanya pada baris train, lalu `transform` ke val |
| Jendela LSTM | Sampel `(WINDOW, fitur)` | LSTM menerima urutan waktu | `make_windows` dengan jendela berurutan |
| Pembuangan baris warm-up | Baris awal tanpa lag/rolling lengkap dan baris akhir tanpa target | Menghindari NaN dan imputasi yang bisa bocor | `dropna()` setelah fitur dibangun |

## Checklist Bagian Hafid
- [ ] Scaler di-fit hanya pada data training (sel Uji Anti-Leakage lulus)
- [ ] Test set tidak dimuat dan tidak disentuh
- [ ] Jumlah sampel tiap split tercatat (`reports/sample_counts_hafid.csv`)
- [ ] Setiap langkah punya alasan tertulis (bagian E), insight EDA diisi
- [ ] `USE_DEMO_DATA = False` sebelum commit, dan notebook dijalankan ulang dari awal sampai akhir
- [ ] Commit: `git add notebooks/02_data_preparation.ipynb` lalu buat pull request untuk di-review Rehan